In [13]:
# Load environment variables from .env
from dotenv import load_dotenv
import os

load_dotenv()

# Check that the Groq API key exists.
# We use bool() so we don't accidentally print the secret key.
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

Groq API key loaded: True


In [14]:
# Web page loading
from langchain_community.document_loaders import WebBaseLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding model
from langchain_huggingface import HuggingFaceEmbeddings

# Vector database
from langchain_chroma import Chroma

# Prompt creation
from langchain_core.prompts import ChatPromptTemplate

# Groq LLM
from langchain_groq import ChatGroq

# Loader

In [15]:
# URL of the document we want our RAG system to answer questions about
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

# WebBaseLoader downloads and converts the webpage into LangChain Documents
loader = WebBaseLoader(url)

# Load the webpage
docs = loader.load()

print("Number of documents:", len(docs))

Number of documents: 1


In [16]:
# Look at the first 1000 characters of the loaded document
print(docs[0].page_content[:1000])







LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
A

In [17]:
print(docs[0].metadata)

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\n\

In [18]:
# Create a text splitter.
#
# chunk_size = maximum approximate size of each chunk
# chunk_overlap = amount of text shared between neighboring chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split the documents into smaller chunks
splits = text_splitter.split_documents(docs)

print("Original documents:", len(docs))
print("Number of chunks:", len(splits))

Original documents: 1
Number of chunks: 66


In [19]:
# Hugging Face embedding model.
#
# It converts text into numerical vectors.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2757.65it/s]


In [20]:
# Convert a sample sentence into an embedding vector
test_vector = embeddings.embed_query("What is self-reflection?")

print("Embedding dimensions:", len(test_vector))

Embedding dimensions: 384


In [21]:
# Create a Chroma vector store.
#
# For every document chunk:
# 1. Generate its embedding
# 2. Store the chunk + embedding in Chroma
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

print("Vector store created!")

Vector store created!


# Retriver

In [22]:
# Convert the Chroma vector store into a retriever.
#
# The retriever's job is to find chunks relevant to a question.
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created!")

Retriever created!


In [23]:
question = "What is self-reflection?"

# Ask the retriever for relevant document chunks
retrieved_docs = retriever.invoke(question)

print("Retrieved documents:", len(retrieved_docs))

Retrieved documents: 4


In [24]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n========== CHUNK {i+1} ==========")
    print(doc.page_content[:700])


========== CHUNK 1 ==========
Illustration of the Reflexion framework. (Image source: Shinn & Labash, 2023)

The heuristic function determines when the trajectory is inefficient or contains hallucination and should be stopped. Inefficient planning refers to trajectories that take too long without success. Hallucination is defined as encountering a sequence of consecutive identical actions that lead to the same observation in the environment.
Self-reflection is created by showing two-shot examples to LLM and each example is a pair of (failed trajectory, ideal reflection for guiding future changes in the plan). Then reflections are added into the agent’s working memory, up to three, to be used as context for querying LLM.


========== CHUNK 2 ==========
Illustration of the Reflexion framework. (Image source: Shinn & Labash, 2023)

The heuristic function determines when the trajectory is inefficient or contains hallucination and should be stopped. Inefficient planning refers to trajector

# LLM

In [33]:
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="qwen/qwen3.6-27b",
    temperature=0
)

print("LLM created!")

LLM created!


In [34]:
# {context} -> That is where our retrieved chunks will go.
system_prompt = """
You are an assistant for question-answering tasks.

Use the following retrieved context to answer the user's question.

If the answer cannot be found in the context, say:
"I don't know based on the provided context."

Keep the answer concise.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# RAG chain

In [29]:
#                  Question
#                     │
#           "What is self-reflection?"
#                     │
#              ┌──────┴──────┐
#              ↓             ↓
#         retriever     Passthrough
#              ↓             ↓
#        relevant          original
#         chunks           question
#              │             │
#              └──────┬──────┘
#                     ↓
#                   prompt
#                     ↓
#                    llm
#                     ↓
#                  answer

In [35]:
#RunnablePassthrough() means:
#Pass the original question to the next component without changing it.
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [36]:
question = "What is task decomposition?"

response = rag_chain.invoke(question)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - Question: "What is task decomposition?"
   - Context provided: Multiple excerpts from a blog post about LLM-powered autonomous agents, specifically focusing on planning and task decomposition.

2.  **Scan Context for Keywords:**
   - Keywords: "task decomposition", "decompose", "subgoals", "break down"
   - Found in Context:
     - "Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks."
     - "Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs."
     - "Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to

In [37]:
question = "What is self-reflection?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs):
    print(f"\n========== SOURCE {i+1} ==========")
    print(doc.page_content[:500])


========== SOURCE 1 ==========
Illustration of the Reflexion framework. (Image source: Shinn & Labash, 2023)

The heuristic function determines when the trajectory is inefficient or contains hallucination and should be stopped. Inefficient planning refers to trajectories that take too long without success. Hallucination is defined as encountering a sequence of consecutive identical actions that lead to the same observation in the environment.
Self-reflection is created by showing two-shot examples to LLM and each example is a

========== SOURCE 2 ==========
Illustration of the Reflexion framework. (Image source: Shinn & Labash, 2023)

The heuristic function determines when the trajectory is inefficient or contains hallucination and should be stopped. Inefficient planning refers to trajectories that take too long without success. Hallucination is defined as encountering a sequence of consecutive identical actions that lead to the same observation in the environment.
Self-reflection is 

# chat-history

In [38]:
# Used to store chat messages
from langchain_core.chat_history import InMemoryChatMessageHistory

# Used to connect chat history with our RAG chain
from langchain_core.runnables.history import RunnableWithMessageHistory

In [39]:
# Dictionary to store chat history for different users/sessions
store = {}

In [40]:
def get_session_history(session_id: str):
    # If this session does not exist,
    # create a new chat history
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    # Return the chat history for this session
    return store[session_id]

In [41]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant.

Use the following context to answer the user's question.

Context:
{context}

If the answer is not present in the context, say you don't know."""
    ),

    # Previous conversation will be inserted here
    MessagesPlaceholder(variable_name="chat_history"),

    # Current user question
    ("human", "{input}")
])

In [44]:
conversational_rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(x["input"])
        ),
        "chat_history": lambda x: x["chat_history"],
        "input": lambda x: x["input"]
    }
    | prompt
    | llm
)

In [46]:
conversational_rag = RunnableWithMessageHistory(
    conversational_rag_chain,

    # Function that returns the history for a session
    get_session_history,

    # Tells the wrapper which input field contains
    # the current user question
    input_messages_key="input",

    # Tells the wrapper where the chat history goes
    history_messages_key="chat_history"
)

c:\Users\sangr\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
response = conversational_rag.invoke(
    {
        "input": "What is task decomposition?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - Question: "What is task decomposition?"
   - Context provided: Multiple paragraphs discussing task decomposition, LLM prompting, LLM+P, Chain of Thought (CoT), Tree of Thoughts (ToT), etc.
   - Constraint: "If the answer is not present in the context, say you don't know."

2.  **Scan Context for Keywords:**
   - Keywords: "task decomposition", "decompose", "steps", "subgoals"
   - Found in context:
     - "Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs."
     - "Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and

In [48]:
response = conversational_rag.invoke(
    {
        "input": "Why is it useful?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "Why is it useful?" referring to "task decomposition" from the previous turn.
2.  **Check Context:** I need to look at the provided context to see if it explains *why* task decomposition is useful.
   - Context provided:
     - LSH description
     - ANNOY description
     - Comparison of MIPS algorithms
     - Component Three: Tool Use (mentions tool use extends model capabilities, sea otter picture)
     - Repeated sections of the above.
   - Wait, the context provided *does not* mention task decomposition at all. It only talks about LSH, ANNOY, MIPS algorithms, and Tool Use.
   - My previous answer about task decomposition was actually generated from my internal knowledge, not the provided context. The prompt says "If the answer is not present in the context, say you don't know." I should have adhered to that constraint in the first turn, but I already answered. Now, for this follow-up, I must strictly fo

In [ ]:
history = get_session_history("user_1")

for message in history.messages:
    print(message.type, ":", message.content)

human : What is task decomposition?
ai : 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - Question: "What is task decomposition?"
   - Context provided: Multiple paragraphs discussing task decomposition, LLM prompting, LLM+P, Chain of Thought (CoT), Tree of Thoughts (ToT), etc.
   - Constraint: "If the answer is not present in the context, say you don't know."

2.  **Scan Context for Keywords:**
   - Keywords: "task decomposition", "decompose", "steps", "subgoals"
   - Found in context:
     - "Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs."
     - "Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation